# CS781 Capstone — Phase 1 — Architecture A: Dense Flash LegalGPT

This notebook is a thin runner for `architecture_a_legalgpt.py`. The actual architecture/training/submission logic lives in the script so the same code can run on Colab, Kaggle, or the RTX 4080 server.

**Architecture:** 12 layers, d=768, 12 MHA heads × 64, SwiGLU 2048, RoPE, Pre-RMSNorm, Flash SDPA, 32K legal Byte-Level BPE, tied embeddings. Nominal size: **110,119,680 parameters**.

In [ ]:
# Upload/extract the project ZIP first if needed, then point PROJECT here.
from pathlib import Path
PROJECT = Path('/content/cs781_phase1_A')
SCRIPT = PROJECT / 'architecture_a_legalgpt.py'
WORKDIR = Path('/content/A0_dense_flash')
print(PROJECT, SCRIPT.exists())

In [ ]:
!pip -q install -r {PROJECT / 'requirements.txt'}

In [ ]:
# Fast architecture/KV-cache unit test
!python {SCRIPT} --mode selftest

## 1. Prepare official data

For the real run this trains the 32K legal tokenizer and writes 200M train + 5M validation tokens to compact uint16 caches. A deterministic document-level holdout prevents train/validation document leakage.

In [ ]:
!python {SCRIPT} --mode prepare \
  --workdir {WORKDIR} \
  --vocab_size 32768 \
  --tokenizer_train_chars 100000000 \
  --prepare_train_tokens 200000000 \
  --prepare_val_tokens 5000000 \
  --val_generation_examples 128 \
  --mix_mode equal

## 2. Train Architecture A

The trainer checkpoints to `latest.pt` and `best.pt` and automatically resumes after interruption.

In [ ]:
!python {SCRIPT} --mode train \
  --workdir {WORKDIR} \
  --max_seq_len 2048 \
  --micro_batch_size 1 \
  --grad_accum_steps 32 \
  --learning_rate 3e-4 \
  --weight_decay 0.1 \
  --target_train_tokens 200000000 \
  --eval_every 100 \
  --save_every 100

## 3. Local generation check

This uses held-out 512-word→128-word examples. By default it reports ROUGE-L and BLEU-4 without downloading an external pretrained scorer. Codabench supplies the authoritative BERTScore.

In [ ]:
!python {SCRIPT} --mode eval \
  --workdir {WORKDIR} \
  --strategy greedy \
  --eval_generation_limit 32

## 4. Build the real Codabench ZIP

This downloads the official Phase-1 test file, generates exactly 128 words for every ID, validates the IDs/schema/word counts, and creates `submission.zip` with `prediction.json` at the ZIP root.

In [ ]:
!python {SCRIPT} --mode submit \
  --workdir {WORKDIR} \
  --strategy greedy

In [ ]:
# Inspect final artifact before upload
import zipfile, json
z = WORKDIR / 'submission' / 'greedy' / 'submission.zip'
print('UPLOAD THIS:', z)
with zipfile.ZipFile(z) as f:
    print('ZIP contents:', f.namelist())
p = WORKDIR / 'submission' / 'greedy' / 'prediction.json'
obj = json.loads(p.read_text())
print('N predictions:', len(obj))
first = next(iter(obj))
print('First ID:', first)
print('First continuation:', obj[first]['text'][:500])

## 5. After Codabench returns scores

Record BERTScore F1, ROUGE-L F1 and BLEU-4 in `experiment_ledger.csv`. Then we make one controlled change at a time (decoding → data/tokenizer → GQA → positional encoding → multi-token prediction) before moving to Architecture E.